In [ ]:
# # Decision Tree Classifier for Weather Prediction
#
# This notebook explores the training, tuning, evaluation, and visual classification performance of a **Decision Tree Classifier** model used to forecast rainfall.
#
# ### Mathematical Concepts & Intuition
# A Decision Tree represents a non-parametric model that builds a binary tree structure by recursively splitting the training datasets.
# At each node, it selects a feature that maximizes **Information Gain** or minimizes **Impurity**.
#
# #### Splitting Criteria Evaluated:
# 1. **Gini Impurity**:
#    $$I_G(p) = 1 - \sum_{i=1}^{J} p_i^2$$
#    Measures how often a randomly chosen element from the set would be incorrectly labeled if it were randomly labeled according to the distribution.
# 2. **Entropy (Information Gain)**:
#    $$H(X) = -\sum_{i=1}^{J} p_i \log_2 p_i$$
#    Measures structural uncertainty or randomness in the target subset.
#
# ### Hyperparameters Tuned
# 1. **Max Depths (`max_depth`)**: Restricts depth of tree (`5`, `10`, `20`, or `None` representing fully grown). Prevents massive structural overfitting.
# 2. **Min Samples Split (`min_samples_split`)**: Minimum instances required inside a node before it can be subdivided (`2`, `5`, or `10`).
# 3. **Criterion**: Gini Impurity vs. Information Gain (Entropy).


In [ ]:
# Step 1: Imports and libraries
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set plotting aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (7, 5)


In [ ]:
# ### Step 2: Data Loading
# Load dataset values from `../data/processed_data.joblib`.


In [ ]:
# Step 2: Load the preprocessed dataset
PROCESSED_DATA_PATH = "../data/processed_data.joblib"
if not os.path.exists(PROCESSED_DATA_PATH):
    raise FileNotFoundError(f"Preprocessed data not found at {PROCESSED_DATA_PATH}")

data = joblib.load(PROCESSED_DATA_PATH)
X_train, X_test, y_train, y_test = data['X_train'], data['X_test'], data['y_train'], data['y_test']
print("Preprocessed data loaded successfully.")


In [ ]:
# ### Step 3: Deep Tuning Loops
# We iterate over depths, minimum splits, and criteria to discover optimal classification limits.


In [ ]:
# Step 3: Training Grid Search
depths = [5, 10, 20, None]
min_splits = [2, 5, 10]
criteria = ['gini', 'entropy']

results = []
best_acc = 0
best_model = None

print("--- Starting Decision Tree Tuning Loop ---")
for depth in depths:
    for split in min_splits:
        for criterion in criteria:
            # Train model
            model = DecisionTreeClassifier(max_depth=depth, min_samples_split=split, 
                                           criterion=criterion, random_state=42)
            model.fit(X_train, y_train)
            
            # Predict and evaluate
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            
            results.append({
                'max_depth': depth,
                'min_samples_split': split,
                'criterion': criterion,
                'accuracy': acc
            })
            
            # Retain best
            if acc > best_acc:
                best_acc = acc
                best_model = model

print("Decision Tree Grid Tuning complete!")
print(f"Best Achieved Test Accuracy: {best_acc * 100:.2f}%")


In [ ]:
# ### Step 4: Metric Reports, Feature Impact & Heatmaps
# Evaluate testing performance, calculate which weather factors have the highest impact, and render a styled heatmap confusion matrix.


In [ ]:
# Step 4: Model evaluation metrics
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=== BEST MODEL METRICS ===")
print(f"Optimal Max Depth:       {best_model.max_depth}")
print(f"Optimal Min Split Size:  {best_model.min_samples_split}")
print(f"Optimal Criterion:       {best_model.criterion}")
print(f"Test Accuracy:           {accuracy * 100:.2f}%")
print(f"Precision Score:         {precision * 100:.2f}%")
print(f"Recall Score:            {recall * 100:.2f}%")
print(f"F1 Performance:          {f1 * 100:.2f}%")

# Calculate Feature Importances (Which features have the most impact on predicting rain?)
feature_names = data.get('feature_names', [f'Feature {i}' for i in range(X_train.shape[1])])
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Print top impactful weather features
print("\n=== MOST IMPACTFUL FEATURES FOR PREDICTING RAIN ===")
for i in range(min(10, len(feature_names))):
    print(f"{i+1}. {feature_names[indices[i]]:<20} | Importance: {importances[indices[i]] * 100:.2f}%")

# Plot Confusion Matrix and Feature Importances side-by-side
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['No Rain', 'Rain'], yticklabels=['No Rain', 'Rain'], ax=axes[0])
axes[0].set_title('Confusion Matrix - Decision Tree (Best Model)', fontsize=13, pad=10)
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

sns.barplot(x=importances[indices[:10]], y=[feature_names[i] for i in indices[:10]], 
            palette='Oranges_r', ax=axes[1])
axes[1].set_title('Top 10 Most Impactful Weather Features', fontsize=13, pad=10)
axes[1].set_xlabel('Relative Importance Score')
axes[1].set_ylabel('Weather Factor')
axes[1].grid(True, linestyle='--')

plt.tight_layout()
plt.show()


In [ ]:
# ### Step 5: Serializing Best Model
# Save to `../data/models/decision_tree.joblib`.


In [ ]:
# Step 5: Save best model
MODELS_DIR = "../data/models"
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

model_path = os.path.join(MODELS_DIR, 'decision_tree.joblib')
joblib.dump(best_model, model_path)
print(f"Best Decision Tree model saved successfully at: {model_path}")
